In [1]:
import re
import compress_fasttext
from tqdm import tqdm
from pathlib import Path
from rapidfuzz import process, fuzz
from rapidfuzz.distance import Levenshtein
import pymorphy3
from utils import get_concepts

In [2]:
# ! curl -L -o fasttext.bin "https://github.com/avidale/compress-fasttext/releases/download/gensim-4-draft/geowac_tokens_sg_300_5_2020-100K-20K-100.bin"

In [3]:
fasttext = compress_fasttext.models.CompressedFastTextKeyedVectors.load('fasttext.bin')
morph = pymorphy3.MorphAnalyzer()

In [4]:
concepts = get_concepts()
concepts[:3]

[('Q1', 'Вселенная'),
 ('Q102836', 'пружина'),
 ('Q104837', 'термодинамическая фаза')]

In [5]:
print('Макисмальное количество слов в понятии (для выбора n-грамм): n =',
      max(map(lambda x: len(x[1].split()), concepts)))

Макисмальное количество слов в понятии (для выбора n-грамм): n = 4


In [6]:
for c in concepts:
    if len(c[1].split()) > 2:
        print(c)

('Q11382', 'закон сохранения энергии')
('Q134465', 'классическая теория тяготения Ньютона')
('Q161635', 'вектор угловой скорости')
('Q192704', 'коэффициент полезного действия')
('Q2305665', 'закон сохранения импульса')
('Q317949', 'активность радиоактивного источника')
('Q483948', 'закон сохранения массы')
('Q649036', 'давление электромагнитного излучения')


In [7]:
def read_concept(concept_id: str):
    with open(Path.cwd().parent / 'advanced_papers' / f'{concept_id}.md', mode='r', encoding='utf-8') as file:
        return '\n'.join(file.readlines())
    

def write_concept(concept_id: str, article: str):
    dir = Path.cwd().parent / 'linked_advanced_papers'
    dir.mkdir(exist_ok=True)
    with open(dir / f'{concept_id}.md', mode='w', encoding='utf-8') as file:
        file.write(article)

In [8]:
def norm(text):
    clean = re.sub('[^\w\s]', ' ', text)
    clean = re.sub('  ', ' ', clean)
    words = clean.split()
    parsed_words = [morph.parse(word)[0] for word in words]
    result = (
        ' '.join([word.normal_form for word in parsed_words]),
        [word.tag.POS for word in parsed_words]
    )
    return result

morphed_concepts = []
for concept_id, concept_name in concepts:
    norm_concept, poses = norm(concept_name)
    morphed_concepts.append((
        concept_id,
        norm_concept,
        poses
    ))
morphed_concepts[:4]

[('Q1', 'вселенная', ['NOUN']),
 ('Q102836', 'пружина', ['NOUN']),
 ('Q104837', 'термодинамический фаза', ['ADJF', 'NOUN']),
 ('Q1075', 'цвет', ['NOUN'])]

In [9]:
def find_related_concept(text: str, except_concept: str):
    """
    text - отрывок (одно, два или три слова) из текста,
    которые предположительно являются концептом. Если
    в text есть лишние слова или отсутствуют слова или
    эти слова не являются концептом, то метод вернёт
    Исходную строку. Иначе метод вернёт корректную
    файловую ссылку в формате Markdown на этот отрывок

    Пример:
    >>> find_related_concept("**Сила**")
    [**Сила**](Q11402.md)

    >>> find_related_concept("силы _Лоренца_")
    [силы _Лоренца_](Q172137.md)

    >>> find_related_concept("сила лоренца присутствует")
    сила лоренца присутствует
    """
    original_text = text[:]

    # Заголовок
    if '#' in text:
        return original_text

    # Некорректный отрывок (слова должны идти подряд, а не через запятую, точку)
    words = text.split()
    if any({'.', ',', ':'} & set(word) for word in words[:-1]):
        return original_text

    # ---

    # Нормализуем вход и создаем сет слов для жесткой проверки
    clean_words = [re.sub(r'[^\w\s]', '', word.strip().lower()) for word in words]
    if any(len(word) == 0 for word in clean_words):
        return original_text
    clean_text = ' '.join(clean_words)
    norm_text, poses_text = norm(clean_text)
    
    # Поиск кандидатов
    results = process.extract(
        norm_text,
        [c[1] for c in morphed_concepts],
        scorer=fuzz.token_sort_ratio,
        score_cutoff=60
    )
    
    answer = []
    for matching, _, id in results:
        poses_match = morphed_concepts[id][2][:]
        if len(poses_text) != len(poses_match):
            continue
        norm_text_splited = norm_text.split()

        matching = matching.split()
        new_matching = []
        f_score = 0
        for w_orig, p_orig in zip(norm_text_splited, poses_text):
            flag = False
            for w_match, p_match in zip(matching, poses_match):
                if p_orig != p_match:
                    continue
                dist = Levenshtein.distance(w_orig, w_match) / len(w_orig)
                if dist > 0.3:
                    continue
                f_score += (1 - dist) / len(norm_text_splited)
                flag = True
                idx = matching.index(w_match)
                del matching[idx]
                del poses_match[idx]
                new_matching.append(w_match)
                break
            if not flag:
                break

        if len(new_matching) != len(norm_text_splited):
            continue

        # Оценка сходства
        similarity = fasttext.similarity(norm_text, ' '.join(new_matching))
        
        # Увеличиваем значимость лексического совпадения над семантическим
        total_score = (f_score * 0.7) + (similarity * 0.3)

        if total_score < 0.75:
            continue

        answer.append((matching, id, total_score))

    if answer:
        # Берем лучший результат
        best_match = max(answer, key=lambda x: x[2])
        id = best_match[1]
        concept = concepts[id][0]
        
        if concept != except_concept:
            return f'[{original_text}]({concept}.md)'
            
    return original_text

In [10]:
print(find_related_concept('физическую систему', ''))
print(find_related_concept('силы Лоренцаа)', ''))

физическую систему
[силы Лоренцаа)](Q172137.md)


In [11]:
def add_links(markdown_text: str, except_concept: str) -> str:
    """
    Проходит по Markdown тексту и добавляет ссылки на концепты,
    используя жадный поиск (3-2-1 слова) и соблюдая границы пунктуации.
    Теперь корректно обрабатывает уже существующий Markdown:
    - [ссылки](...) и ![изображения](...) — полностью пропускаются (даже если внутри есть слова-концепты).
    - **жирный** / __жирный__ / *курсив* / _курсив_ — проверяются целиком как один кандидат
      (find_related_concept получает весь кусок с разметкой).
    """
    lines = markdown_text.split('\n')
    processed_lines = []
    for line in lines:
        # Правило 1: Не искать ссылки в заголовках
        if line.lstrip().startswith('#'):
            processed_lines.append(line)
            continue

        # === НОВОЕ: Защита Markdown-элементов ===
        md_matches = []
        counter = 0

        def replacer(match):
            nonlocal counter
            counter += 1
            md = match.group(0)
            ph = f"__MD_{counter}__"
            is_link = re.match(r"^!?\[", md) is not None
            md_matches.append({"original": md, "ph": ph, "is_link": is_link})
            return ph

        # Regex ловит:
        # 1. ссылки и картинки (с ( ) внутри — поэтому защита обязательна)
        # 2. **bold**, __bold__, *italic*, _italic_
        #    (используем negative lookahead, чтобы не ломаться на вложенных)
        md_re = re.compile(
            r'(!?\[[^\]]+\]\([^)]+\))'
            r'|\*\*(?:(?!\*\*).)*?\*\*'
            r'|__(?:(?!__).)*?__'
            r'|\*(?:(?!\*).)*?\*'
            r'|_(?:(?!_).)*?_'
        )
        protected_line = md_re.sub(replacer, line)
        # ========================================

        # Оригинальная обработка (теперь на защищённой строке)
        # Правило 2 & 4 & 5: Границы (.,:;). Разделяем, сохраняя сами разделители.
        parts = re.split(r'([.,:;!?()])', protected_line)
        new_parts = []
        for part in parts:
            if part in '.,:;':
                new_parts.append(part)
            else:
                # Обрабатываем текст внутри «безопасного» сегмента
                new_parts.append(_process_segment(part, except_concept))
        processed_line = "".join(new_parts)

        # === ВОССТАНОВЛЕНИЕ Markdown ===
        for info in md_matches:
            ph = info["ph"]
            if info["is_link"]:
                replacement = info["original"]          # ссылки/картинки — без изменений
            else:
                # **жирный** / *курсив* — проверяем целиком
                replacement = find_related_concept(info["original"], except_concept)
            processed_line = processed_line.replace(ph, replacement, 1)
        # ========================================

        processed_lines.append(processed_line)

    return re.sub('\n\n', '\n', "\n".join(processed_lines))

def _process_segment(segment: str, except_concept: str) -> str:
    if not segment.strip():
        return segment

    # Регулярное выражение для захвата:
    # 1. [текст](ссылка) - группа 1
    # 2. **текст** или *текст* - группа 2
    # 3. Пробелы - группа 3
    # Все остальное попадет в split как обычные слова
    pattern = r'(\[.*?\]\(.*?\))|(\*{1,2}.+?\*{1,2})|(\s+)'
    
    # Разбиваем сегмент, сохраняя все группы
    raw_tokens = re.split(pattern, segment)
    
    # Очищаем от None (возникают из-за несовпавших групп в re.split)
    tokens = [t for t in raw_tokens if t is not None and t != '']
    
    # Определяем, какие токены можно пытаться линковать.
    # Мы линкуем: обычные слова и текст в звездах (группа 2).
    # Мы НЕ линкуем: существующие ссылки (группа 1) и пробелы (группа 3).
    
    linkable_indices = []
    for i, token in enumerate(tokens):
        # Проверяем, не является ли токен уже ссылкой
        if re.match(r'\[.*?\]\(.*?\)', token):
            continue
        # Проверяем, не пробел ли это
        if token.strip() == '':
            continue
        # Все остальное (слова и **жирный**) можно подавать в find_related_concept
        linkable_indices.append(i)

    result_fragments = []
    last_processed_token_idx = 0
    p = 0
    
    while p < len(linkable_indices):
        found_match = False
        
        # Жадный поиск по индексам, которые разрешено линковать
        for window_size in [3, 2, 1]:
            if p + window_size <= len(linkable_indices):
                # Проверяем, что между токенами в окне нет «запрещенных» элементов (существующих ссылок)
                # Вычисляем физический диапазон в списке tokens
                start_tok_idx = linkable_indices[p]
                end_tok_idx = linkable_indices[p + window_size - 1]
                
                # Проверка: нет ли внутри этого диапазона токенов, которые не в списке linkable_indices
                # (кроме пробелов, которые допустимы внутри понятия)
                actual_slice = tokens[start_tok_idx : end_tok_idx + 1]
                
                # Если в слайсе есть уже готовая ссылка — это окно невалидно
                if any(re.match(r'\[.*?\]\(.*?\)', t) for t in actual_slice):
                    continue

                candidate = "".join(actual_slice)
                linked = find_related_concept(candidate, except_concept)
                
                if linked != candidate:
                    # Добавляем все токены, которые были пропущены до начала этого окна
                    result_fragments.append("".join(tokens[last_processed_token_idx : start_tok_idx]))
                    # Добавляем саму ссылку
                    result_fragments.append(linked)
                    
                    last_processed_token_idx = end_tok_idx + 1
                    p += window_size
                    found_match = True
                    break
        
        if not found_match:
            # Если ничего не нашли, просто двигаем указатель
            curr_idx = linkable_indices[p]
            result_fragments.append("".join(tokens[last_processed_token_idx : curr_idx + 1]))
            last_processed_token_idx = curr_idx + 1
            p += 1
            
    # Добавляем хвост
    result_fragments.append("".join(tokens[last_processed_token_idx:]))
    
    return "".join(result_fragments)

In [12]:
for concept_id, concept_name in tqdm(concepts):
    article = read_concept(concept_id)
    new_article = add_links(article, except_concept=concept_id)
    write_concept(concept_id, new_article)

100%|██████████| 168/168 [03:59<00:00,  1.42s/it]
